# Tree-Ring Watermarks: Baseline vs Optimized Comparison

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashhad1130/Invisible_Watermark_GNN/blob/main/Google_Colab_TreeRing_Watermark.ipynb)

This notebook runs the full Tree-Ring Watermark pipeline on **Google Colab (CUDA)** or **M4 Pro (MPS)**.

**Paper:** [Tree-Ring Watermarks: Fingerprints for Diffusion Images that are Invisible and Robust](https://arxiv.org/abs/2305.20030)

## What this notebook does:
1. Installs all dependencies
2. Clones the original tree-ring-watermark repo
3. Runs the **baseline** experiment (original paper settings)
4. Runs the **optimized** experiment (multi-channel, larger radius, scaled injection)
5. Compares results with plots

## Usage:
- Set `SCALE = "small"` for quick verification (10 images, ~15 min on T4)
- Set `SCALE = "large"` for full benchmark (100 images, ~2.5 hrs on T4)

---
## Cell 1: Configuration

In [ ]:
#@title Configuration {display-mode: "form"}
#@markdown ### Experiment Settings
SCALE = "small"  #@param ["small", "large"]
SKIP_CLIP = True  #@param {type:"boolean"}
SCALE_FACTOR = 1.1  #@param {type:"number"}

import os
PROJECT_DIR = os.getcwd()
print(f"Project dir: {PROJECT_DIR}")
print(f"Scale: {SCALE} | Skip CLIP: {SKIP_CLIP} | Scale factor: {SCALE_FACTOR}")

---
## Cell 2: Install Dependencies & Clone Repo

In [ ]:
%%bash
# Install dependencies
# Note: torch and torchvision are pre-installed on Colab; we only add the diffusion packages.
pip install -q diffusers==0.21.4 transformers==4.35.2 accelerate safetensors \
    "huggingface-hub>=0.19,<0.25" datasets scipy scikit-learn Pillow matplotlib tqdm "numpy<2.0"

# Clone tree-ring-watermark repo if not present
if [ ! -d "tree-ring-watermark" ]; then
    git clone --depth 1 https://github.com/YuxinWenRick/tree-ring-watermark.git
    echo "Cloned tree-ring-watermark repo"
else
    echo "tree-ring-watermark/ already exists"
fi

mkdir -p experiment/checkpoints experiment/results experiment/outputs
echo "Setup complete."

---
## Cell 3: Check GPU & Environment

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("MPS (Apple Silicon)")
else:
    DEVICE = "cpu"
    print("WARNING: No GPU found. This will be very slow.")

import diffusers, transformers
print(f"diffusers: {diffusers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"Device: {DEVICE}")

---
## Cell 4: Write Experiment Code

This cell writes all experiment Python files. The original `tree-ring-watermark/` repo is **never modified**.

In [ ]:
import os

os.makedirs("experiment", exist_ok=True)

# ===== configs.py =====
with open("experiment/configs.py", "w") as f:
    f.write('''"""Experiment configurations for baseline vs optimized comparison."""
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class WatermarkConfig:
    w_seed: int = 999999
    w_channel: int = 3
    w_pattern: str = "ring"
    w_mask_shape: str = "circle"
    w_radius: int = 10
    w_measurement: str = "l1_complex"
    w_injection: str = "complex"
    w_pattern_const: float = 0.0

@dataclass
class AttackConfig:
    name: str = "no_attack"
    r_degree: Optional[float] = None
    jpeg_ratio: Optional[int] = None
    crop_scale: Optional[float] = None
    crop_ratio: Optional[float] = None
    gaussian_blur_r: Optional[int] = None
    gaussian_std: Optional[float] = None
    brightness_factor: Optional[float] = None
    rand_aug: int = 0

@dataclass
class ExperimentConfig:
    name: str = "experiment"
    approach: str = "baseline"
    model_id: str = "runwayml/stable-diffusion-v1-5"
    image_length: int = 512
    num_images: int = 1
    guidance_scale: float = 7.5
    num_inference_steps: int = 50
    test_num_inference_steps: Optional[int] = None
    gen_seed: int = 0
    start: int = 0
    end: int = 10
    reference_model: Optional[str] = "ViT-g-14"
    reference_model_pretrain: Optional[str] = "laion2b_s12b_b42k"
    watermark: WatermarkConfig = field(default_factory=WatermarkConfig)
    attacks: List[AttackConfig] = field(default_factory=list)
    checkpoint_every: int = 5
    checkpoint_dir: str = "checkpoints"
    results_dir: str = "results"
    outputs_dir: str = "outputs"

    def __post_init__(self):
        if self.test_num_inference_steps is None:
            self.test_num_inference_steps = self.num_inference_steps

BASIC_ATTACKS = [
    AttackConfig(name="no_attack"),
    AttackConfig(name="rotation_75", r_degree=75.0),
    AttackConfig(name="jpeg_25", jpeg_ratio=25),
    AttackConfig(name="crop_0.75", crop_scale=0.75, crop_ratio=0.75),
    AttackConfig(name="gaussian_blur_4", gaussian_blur_r=4),
    AttackConfig(name="gaussian_noise_0.1", gaussian_std=0.1),
    AttackConfig(name="brightness_6", brightness_factor=6.0),
]

EXTENDED_ATTACKS = BASIC_ATTACKS + [
    AttackConfig(name="rotation_45", r_degree=45.0),
    AttackConfig(name="jpeg_50", jpeg_ratio=50),
    AttackConfig(name="crop_0.5", crop_scale=0.5, crop_ratio=0.5),
    AttackConfig(name="gaussian_blur_8", gaussian_blur_r=8),
    AttackConfig(name="gaussian_noise_0.05", gaussian_std=0.05),
    AttackConfig(name="brightness_3", brightness_factor=3.0),
]

def get_small_scale_baseline():
    return ExperimentConfig(name="small_baseline", approach="baseline", start=0, end=10,
        watermark=WatermarkConfig(w_channel=3, w_pattern="ring", w_radius=10),
        attacks=BASIC_ATTACKS, checkpoint_every=5)

def get_small_scale_optimized():
    return ExperimentConfig(name="small_optimized", approach="optimized", start=0, end=10,
        num_inference_steps=50, test_num_inference_steps=75,
        watermark=WatermarkConfig(w_channel=-1, w_pattern="ring", w_radius=14),
        attacks=BASIC_ATTACKS, checkpoint_every=5)

def get_large_scale_baseline():
    return ExperimentConfig(name="large_baseline", approach="baseline", start=0, end=100,
        watermark=WatermarkConfig(w_channel=3, w_pattern="ring", w_radius=10),
        attacks=EXTENDED_ATTACKS, checkpoint_every=10)

def get_large_scale_optimized():
    return ExperimentConfig(name="large_optimized", approach="optimized", start=0, end=100,
        num_inference_steps=50, test_num_inference_steps=75,
        watermark=WatermarkConfig(w_channel=-1, w_pattern="ring", w_radius=14),
        attacks=EXTENDED_ATTACKS, checkpoint_every=10)
''')

print("  configs.py written")

In [ ]:
# ===== landscape_prompts.py =====
with open("experiment/landscape_prompts.py", "w") as f:
    f.write('''"""Curated landscape/scenery prompts for watermark experiments."""

LANDSCAPE_PROMPTS = [
    "A breathtaking panoramic view of snow-capped mountains at golden hour with a crystal clear alpine lake in the foreground",
    "Misty mountain peaks emerging from clouds at dawn, with pine forests covering the lower slopes",
    "A dramatic mountain landscape with a winding river valley, autumn colors on the hillsides",
    "Towering granite cliffs reflected perfectly in a still mountain lake surrounded by wildflowers",
    "A serene alpine meadow with snow-capped peaks in the background under a clear blue sky",
    "An enchanted old-growth forest with sunbeams filtering through ancient moss-covered trees",
    "A peaceful birch forest in autumn with golden leaves carpeting the ground and soft morning light",
    "Dense bamboo forest with shafts of light creating patterns on the forest floor",
    "A redwood forest scene with massive tree trunks and ferns in the misty understory",
    "A winding path through a magical forest with colorful wildflowers and dappled sunlight",
    "A dramatic seascape with waves crashing against rocky cliffs at sunset with vibrant orange and purple sky",
    "A tranquil tropical beach with crystal clear turquoise water and palm trees swaying in the breeze",
    "Rugged coastal cliffs overlooking a deep blue ocean with seabirds soaring overhead",
    "A peaceful cove with emerald green water surrounded by limestone karst formations",
    "Golden hour over a calm ocean with gentle waves lapping on a sandy shore",
    "A vast desert landscape with towering red sandstone formations under a star-filled night sky",
    "Grand Canyon at sunset with layers of colorful rock strata and deep shadows",
    "Rolling sand dunes in the Sahara desert with beautiful patterns created by wind",
    "A desert oasis with palm trees and a small pond surrounded by golden sand dunes",
    "Dramatic desert mesa formations with a thunderstorm approaching in the distance",
    "A mirror-like lake at dawn reflecting autumn trees and mountains in perfect symmetry",
    "A cascading waterfall flowing into a turquoise pool surrounded by lush tropical vegetation",
    "A gentle river winding through a valley of wildflowers with mountains in the distance",
    "A frozen lake surrounded by snow-covered pine trees under northern lights aurora borealis",
    "A peaceful pond with lily pads and lotus flowers surrounded by weeping willows at sunset",
    "Endless lavender fields in Provence stretching to the horizon under a golden sunset sky",
    "A rolling countryside with green hills, stone walls, and scattered wildflowers in spring",
    "A sunflower field at golden hour with warm light illuminating thousands of blooms",
    "Terraced rice paddies reflecting the sunset sky with mountains in the background",
    "A vast prairie grassland with dramatic cumulus clouds and golden afternoon light",
    "A frozen waterfall with blue ice formations and snow-covered evergreen trees",
    "A winter wonderland with fresh snow covering a peaceful village and surrounding mountains",
    "Ice caves with stunning blue ice formations and light filtering through from above",
    "A snow-covered mountain pass with a lone traveler and dramatic cloud formations",
    "Arctic tundra landscape with icebergs floating in calm waters under midnight sun",
    "Lush tropical rainforest with a hidden waterfall and exotic birds and butterflies",
    "A volcanic landscape with lava fields and a smoking crater under dramatic skies",
    "Terraced hillsides with tea plantations stretching into misty mountain valleys",
    "A coral reef visible through crystal clear shallow water with a tropical island behind",
    "Dense jungle canopy seen from above with a river snaking through the green expanse",
    "A dramatic sunset over rolling hills with layers of clouds painted in red and gold",
    "A double rainbow arching over a lush green valley after a summer rainstorm",
    "Northern lights dancing over a snowy mountain landscape with a clear starry sky",
    "Dramatic cumulonimbus clouds towering over a peaceful countryside at golden hour",
    "A foggy morning in a valley with treetops emerging from the mist like islands",
    "A traditional Japanese zen garden with raked gravel, moss, and carefully pruned trees",
    "An English cottage garden overflowing with roses, foxgloves, and delphiniums in summer",
    "A cherry blossom avenue in full bloom with pink petals falling like snow",
    "A formal French garden with geometric hedges, fountains, and symmetrical flower beds",
    "A wild garden meadow with poppies, cornflowers, and butterflies in warm afternoon light",
    "Dramatic fjord landscape with steep cliffs and deep blue water under overcast skies",
    "A vast savanna with acacia trees silhouetted against a spectacular African sunset",
    "Limestone karst mountains rising from emerald rice paddies in morning mist",
    "A hidden valley with waterfalls cascading down mossy cliffs into a turquoise pool",
    "Ancient sequoia grove with massive trees and dappled sunlight on the forest floor",
    "A coastal marsh at low tide with golden grasses and reflective tidal pools",
    "Snow-dusted volcanic peaks rising above a sea of clouds at sunrise",
    "A winding mountain road through autumn foliage with fog in the valley below",
    "Glacial valley with U-shaped walls, a braided river, and distant ice fields",
    "A Mediterranean hillside village surrounded by olive groves and cypress trees at sunset",
    "Turquoise hot springs terraces with steam rising against a backdrop of mountains",
    "A field of tulips in perfect rows stretching to a windmill on the horizon",
    "Dramatic badlands erosion patterns in striped sedimentary rock under a stormy sky",
    "A peaceful monastery perched on a cliff overlooking a misty mountain valley",
    "Bioluminescent bay at night with glowing blue water and a starry sky above",
    "A sprawling vineyard in autumn with golden and red leaves and rolling hills beyond",
    "Crystal clear cenote with light beams penetrating deep turquoise water surrounded by jungle",
    "A vast flower field with mixed wildflowers stretching to distant blue mountains",
    "Majestic redrock canyon with a narrow slot revealing blue sky above",
    "A tranquil bamboo grove with a stone path and soft diffused green light",
    "Spectacular icefall on a glacier with blue crevasses and mountain peaks behind",
    "A highland plateau with grazing yaks and prayer flags against Himalayan peaks",
    "Tropical waterfall surrounded by giant ferns and orchids in a cloud forest",
    "A peaceful countryside with a stone bridge over a stream and rolling green hills",
    "Dramatic sea stacks and arches along a rugged coastline at low tide sunset",
    "An autumn forest reflected in a perfectly still beaver pond at dawn",
    "A desert canyon at night with the Milky Way spanning the sky overhead",
    "Lush terraced hillsides with cascading waterfalls in a tropical paradise",
    "A snow-covered boreal forest stretching to the horizon under pink twilight sky",
    "Dramatic thunder clouds over a golden wheat field with a lone tree",
    "A coral atoll seen from above with rings of turquoise and deep blue water",
    "Misty Scottish highlands with heather-covered hills and a distant loch",
    "A bamboo raft on a emerald river winding through towering limestone peaks",
    "Spring cherry blossoms framing a view of a snow-capped volcano",
    "An ancient stone circle on a windswept moorland under dramatic skies",
    "A tropical sunset over calm waters with silhouetted palm trees and fishing boats",
    "Vast salt flats creating a perfect mirror reflection of clouds and mountains",
    "A hidden grotto with crystal stalactites reflecting in an underground lake",
    "Rolling Tuscan hills with cypress-lined roads and golden wheat fields at sunset",
    "A dramatic waterfall plunging into a deep gorge surrounded by rainforest",
    "Arctic ice shelf edge with turquoise icebergs and a polar bear in the distance",
]

def get_prompt(index):
    return LANDSCAPE_PROMPTS[index % len(LANDSCAPE_PROMPTS)]

def get_prompts(start, end):
    return [get_prompt(i) for i in range(start, end)]
''')

print("  landscape_prompts.py written")

In [ ]:
# ===== device_compat.py (handles CUDA + MPS + CPU + JPEG fix) =====
DEVICE_COMPAT_CODE = """
"""Device compatibility layer: handles FFT dtype issues on MPS, passthrough on CUDA/CPU.
Also patches image_distortion to fix lazy-load JPEG bug in newer Pillow."""
import sys, io
from pathlib import Path
import torch
from PIL import Image

REPO_ROOT = Path(__file__).resolve().parent.parent / "tree-ring-watermark"
sys.path.insert(0, str(REPO_ROOT))
import optim_utils as _ou

def get_device():
    if torch.cuda.is_available(): return "cuda"
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available(): return "mps"
    return "cpu"

def _needs_fp32_fft(device):
    return device == "mps"

def get_watermarking_pattern(pipe, args, device, shape=None):
    if _needs_fp32_fft(device):
        orig_dtype = pipe.text_encoder.dtype
        pipe.text_encoder = pipe.text_encoder.float()
        result = _ou.get_watermarking_pattern(pipe, args, device, shape)
        pipe.text_encoder = pipe.text_encoder.to(orig_dtype)
        return result
    return _ou.get_watermarking_pattern(pipe, args, device, shape)

def inject_watermark(init_latents_w, watermarking_mask, gt_patch, args):
    if _needs_fp32_fft(init_latents_w.device.type):
        orig_dtype = init_latents_w.dtype
        result = _ou.inject_watermark(init_latents_w.float(), watermarking_mask, gt_patch, args)
        return result.to(orig_dtype)
    return _ou.inject_watermark(init_latents_w, watermarking_mask, gt_patch, args)

def eval_watermark(reversed_latents_no_w, reversed_latents_w, watermarking_mask, gt_patch, args):
    if _needs_fp32_fft(reversed_latents_no_w.device.type):
        return _ou.eval_watermark(
            reversed_latents_no_w.float(), reversed_latents_w.float(),
            watermarking_mask, gt_patch, args)
    return _ou.eval_watermark(reversed_latents_no_w, reversed_latents_w, watermarking_mask, gt_patch, args)

# Patch: fix JPEG lazy-load bug in image_distortion.
_orig_image_distortion = _ou.image_distortion

def _patched_image_distortion(img1, img2, seed, args):
    if args.jpeg_ratio is not None:
        buf1 = io.BytesIO()
        img1.save(buf1, format="JPEG", quality=args.jpeg_ratio)
        buf1.seek(0)
        img1 = Image.open(buf1)
        img1.load()

        buf2 = io.BytesIO()
        img2.save(buf2, format="JPEG", quality=args.jpeg_ratio)
        buf2.seek(0)
        img2 = Image.open(buf2)
        img2.load()

        orig_ratio = args.jpeg_ratio
        args.jpeg_ratio = None
        img1, img2 = _orig_image_distortion(img1, img2, seed, args)
        args.jpeg_ratio = orig_ratio
        return img1, img2
    return _orig_image_distortion(img1, img2, seed, args)

_ou.image_distortion = _patched_image_distortion

"""

with open("experiment/device_compat.py", "w") as f:
    f.write(DEVICE_COMPAT_CODE)

print("  device_compat.py written")

In [ ]:
# ===== run_baseline.py =====
with open("experiment/run_baseline.py", "w") as f:
    f.write('''"""Baseline runner: uses ORIGINAL tree-ring code with landscape prompts + checkpointing."""
import sys, os, json, time, copy, argparse
from pathlib import Path
from tqdm import tqdm
from statistics import mean
from sklearn import metrics
import torch, numpy as np

REPO_ROOT = Path(__file__).resolve().parent.parent / "tree-ring-watermark"
sys.path.insert(0, str(REPO_ROOT))
from inverse_stable_diffusion import InversableStableDiffusionPipeline
from diffusers import DPMSolverMultistepScheduler
from optim_utils import (set_random_seed, transform_img, get_watermarking_mask,
                         image_distortion, measure_similarity)

sys.path.insert(0, str(Path(__file__).resolve().parent))
from device_compat import get_device, get_watermarking_pattern, inject_watermark, eval_watermark
from landscape_prompts import get_prompt
from configs import ExperimentConfig, AttackConfig

def build_args(config, attack):
    a = argparse.Namespace()
    a.model_id=config.model_id; a.image_length=config.image_length; a.num_images=config.num_images
    a.guidance_scale=config.guidance_scale; a.num_inference_steps=config.num_inference_steps
    a.test_num_inference_steps=config.test_num_inference_steps; a.gen_seed=config.gen_seed
    a.start=config.start; a.end=config.end; a.run_name=f"{config.name}_{attack.name}"
    a.with_tracking=False; a.max_num_log_image=100
    wm=config.watermark
    a.w_seed=wm.w_seed; a.w_channel=wm.w_channel; a.w_pattern=wm.w_pattern
    a.w_mask_shape=wm.w_mask_shape; a.w_radius=wm.w_radius; a.w_measurement=wm.w_measurement
    a.w_injection=wm.w_injection; a.w_pattern_const=wm.w_pattern_const
    a.r_degree=attack.r_degree; a.jpeg_ratio=attack.jpeg_ratio; a.crop_scale=attack.crop_scale
    a.crop_ratio=attack.crop_ratio; a.gaussian_blur_r=attack.gaussian_blur_r
    a.gaussian_std=attack.gaussian_std; a.brightness_factor=attack.brightness_factor; a.rand_aug=attack.rand_aug
    a.dataset="Gustavosta/Stable-Diffusion-Prompts"
    a.reference_model=config.reference_model; a.reference_model_pretrain=config.reference_model_pretrain
    return a

def load_ckpt(p):
    if p.exists():
        with open(p) as f: return json.load(f)
    return None

def save_ckpt(p, data):
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p,"w") as f: json.dump(data, f, indent=2)

def run_baseline(config, skip_clip=False):
    device = get_device()
    dtype = torch.float16 if device == "cuda" else torch.float16  # fp16 for both CUDA and MPS

    print(f"\\n{\'=\'*60}\\nBASELINE: {config.name} | Device: {device} | Images: {config.start}-{config.end}")
    print(f"Watermark: ch={config.watermark.w_channel}, pattern={config.watermark.w_pattern}, r={config.watermark.w_radius}\\n{\'=\'*60}\\n")

    base_dir = Path(__file__).resolve().parent
    ckpt_dir = base_dir/config.checkpoint_dir/config.name
    res_dir = base_dir/config.results_dir/config.name
    out_dir = base_dir/config.outputs_dir/config.name
    for d in [ckpt_dir, res_dir, out_dir]: d.mkdir(parents=True, exist_ok=True)

    args = build_args(config, config.attacks[0] if config.attacks else AttackConfig())

    print("Loading Stable Diffusion pipeline...")
    scheduler = DPMSolverMultistepScheduler.from_pretrained(args.model_id, subfolder="scheduler")
    pipe = InversableStableDiffusionPipeline.from_pretrained(args.model_id, scheduler=scheduler, torch_dtype=dtype)
    pipe = pipe.to(device)

    ref_model=ref_clip_preprocess=ref_tokenizer=None
    if not skip_clip and config.reference_model:
        print(f"Loading CLIP: {config.reference_model}...")
        import open_clip
        ref_model, _, ref_clip_preprocess = open_clip.create_model_and_transforms(config.reference_model, pretrained=config.reference_model_pretrain, device=device)
        ref_tokenizer = open_clip.get_tokenizer(config.reference_model)

    text_embeddings = pipe.get_text_embedding("")
    gt_patch = get_watermarking_pattern(pipe, args, device)

    all_results = {}
    for attack in config.attacks:
        args_a = build_args(config, attack)
        ckpt_path = ckpt_dir/f"{attack.name}_checkpoint.json"
        ckpt = load_ckpt(ckpt_path)

        if ckpt and ckpt.get("completed"):
            print(f"  {attack.name}: already done, loading.")
            all_results[attack.name] = ckpt["final_results"]; continue

        start_idx=config.start; results=[]; no_w_metrics=[]; w_metrics=[]; clips=[]; clips_w=[]; saved=[]
        if ckpt and not ckpt.get("completed"):
            start_idx=ckpt["last_idx"]+1; results=ckpt.get("results",[]); no_w_metrics=ckpt.get("no_w_metrics",[])
            w_metrics=ckpt.get("w_metrics",[]); clips=ckpt.get("clips",[]); clips_w=ckpt.get("clips_w",[]); saved=ckpt.get("saved",[])
            print(f"  {attack.name}: resuming from {start_idx}")

        t0=time.time()
        for i in tqdm(range(start_idx, config.end), desc=f"  {attack.name}"):
            seed=i+config.gen_seed; prompt=get_prompt(i)
            set_random_seed(seed)
            lat_no_w=pipe.get_random_latents()
            out_no_w=pipe(prompt,num_images_per_prompt=1,guidance_scale=args_a.guidance_scale,num_inference_steps=args_a.num_inference_steps,height=args_a.image_length,width=args_a.image_length,latents=lat_no_w)
            img_no_w=out_no_w.images[0]

            lat_w=copy.deepcopy(lat_no_w)
            mask=get_watermarking_mask(lat_w, args_a, device)
            lat_w=inject_watermark(lat_w, mask, gt_patch, args_a)
            out_w=pipe(prompt,num_images_per_prompt=1,guidance_scale=args_a.guidance_scale,num_inference_steps=args_a.num_inference_steps,height=args_a.image_length,width=args_a.image_length,latents=lat_w)
            img_w=out_w.images[0]

            if len(saved)<5:
                d=out_dir/attack.name; d.mkdir(parents=True,exist_ok=True)
                img_no_w.save(d/f"img_{i:04d}_no_wm.png"); img_w.save(d/f"img_{i:04d}_wm.png"); saved.append(i)

            img_no_w_a, img_w_a = image_distortion(img_no_w, img_w, seed, args_a)

            t_no=transform_img(img_no_w_a).unsqueeze(0).to(text_embeddings.dtype).to(device)
            rev_no=pipe.forward_diffusion(latents=pipe.get_image_latents(t_no,sample=False),text_embeddings=text_embeddings,guidance_scale=1,num_inference_steps=args_a.test_num_inference_steps)
            t_w=transform_img(img_w_a).unsqueeze(0).to(text_embeddings.dtype).to(device)
            rev_w=pipe.forward_diffusion(latents=pipe.get_image_latents(t_w,sample=False),text_embeddings=text_embeddings,guidance_scale=1,num_inference_steps=args_a.test_num_inference_steps)

            nm,wm_=eval_watermark(rev_no,rev_w,mask,gt_patch,args_a)
            cs_no=cs_w=0.0
            if ref_model:
                sims=measure_similarity([img_no_w,img_w],prompt,ref_model,ref_clip_preprocess,ref_tokenizer,device)
                cs_no=sims[0].item(); cs_w=sims[1].item()

            results.append({"i":i,"prompt":prompt,"no_w":nm,"w":wm_,"clip_no":cs_no,"clip_w":cs_w})
            no_w_metrics.append(-nm); w_metrics.append(-wm_); clips.append(cs_no); clips_w.append(cs_w)

            if (i-config.start+1)%config.checkpoint_every==0 or i==config.end-1:
                save_ckpt(ckpt_path,{"attack":attack.name,"last_idx":i,"completed":False,"results":results,
                    "no_w_metrics":no_w_metrics,"w_metrics":w_metrics,"clips":clips,"clips_w":clips_w,"saved":saved})

        elapsed=time.time()-t0
        preds=no_w_metrics+w_metrics; labels=[0]*len(no_w_metrics)+[1]*len(w_metrics)
        fpr,tpr,_=metrics.roc_curve(labels,preds,pos_label=1)
        auc=metrics.auc(fpr,tpr); acc=np.max(1-(fpr+(1-tpr))/2)
        low=tpr[np.where(fpr<.01)[0][-1]] if len(np.where(fpr<.01)[0])>0 else 0.0

        final={"attack":attack.name,"approach":"baseline","num_images":config.end-config.start,
            "auc":float(auc),"acc":float(acc),"tpr_at_1fpr":float(low),
            "mean_no_w_metric":float(mean([-m for m in no_w_metrics])),"mean_w_metric":float(mean([-m for m in w_metrics])),
            "clip_score_mean":float(mean(clips)) if clips and any(c!=0 for c in clips) else None,
            "clip_score_w_mean":float(mean(clips_w)) if clips_w and any(c!=0 for c in clips_w) else None,
            "elapsed_seconds":elapsed,
            "watermark_params":{"w_channel":config.watermark.w_channel,"w_pattern":config.watermark.w_pattern,"w_radius":config.watermark.w_radius}}
        all_results[attack.name]=final
        save_ckpt(ckpt_path,{"attack":attack.name,"last_idx":config.end-1,"completed":True,"results":results,
            "no_w_metrics":no_w_metrics,"w_metrics":w_metrics,"clips":clips,"clips_w":clips_w,"saved":saved,"final_results":final})
        print(f"  AUC:{auc:.4f} Acc:{acc:.4f} TPR@1%FPR:{low:.4f} Time:{elapsed:.1f}s")

    with open(res_dir/"all_attacks_results.json","w") as f: json.dump(all_results,f,indent=2)
    print(f"\\nBaseline results: {res_dir/\'all_attacks_results.json\'}")
    return all_results

if __name__=="__main__":
    import argparse as ap; p=ap.ArgumentParser(); p.add_argument("--scale",default="small"); p.add_argument("--skip_clip",action="store_true"); a=p.parse_args()
    from configs import get_small_scale_baseline, get_large_scale_baseline
    run_baseline(get_small_scale_baseline() if a.scale=="small" else get_large_scale_baseline(), skip_clip=a.skip_clip)
''')

print("  run_baseline.py written")

In [ ]:
# ===== run_optimized.py =====
with open("experiment/run_optimized.py", "w") as f:
    f.write('''"""Optimized runner. Imports from original repo without modification.
Optimizations:
  1. Multi-channel watermarking (w_channel=-1): all 4 latent channels
  2. Larger radius (w_radius=14): more frequency coefficients
  3. More inversion steps (test_steps=75): better DDIM inversion accuracy
  4. Amplitude-scaled injection (1.1x): stronger watermark energy
  5. Ensemble per-channel detection: robust to channel-specific attacks
"""
import sys, os, json, time, copy, argparse
from pathlib import Path
from tqdm import tqdm
from statistics import mean
from sklearn import metrics
import torch, numpy as np

REPO_ROOT = Path(__file__).resolve().parent.parent / "tree-ring-watermark"
sys.path.insert(0, str(REPO_ROOT))
from inverse_stable_diffusion import InversableStableDiffusionPipeline
from diffusers import DPMSolverMultistepScheduler
from optim_utils import (set_random_seed, transform_img, get_watermarking_mask,
                         image_distortion, measure_similarity)

sys.path.insert(0, str(Path(__file__).resolve().parent))
from device_compat import get_device, get_watermarking_pattern, eval_watermark as _eval_base
from landscape_prompts import get_prompt
from configs import ExperimentConfig, AttackConfig

# Optimization #4: Amplitude-scaled injection
def inject_watermark_scaled(init_latents_w, watermarking_mask, gt_patch, args, scale_factor=1.1):
    orig_dtype = init_latents_w.dtype
    init_latents_w = init_latents_w.float()
    init_latents_w_fft = torch.fft.fftshift(torch.fft.fft2(init_latents_w), dim=(-1, -2))
    scaled_patch = gt_patch.clone()
    scaled_patch[watermarking_mask] = gt_patch[watermarking_mask] * scale_factor
    if args.w_injection == "complex":
        init_latents_w_fft[watermarking_mask] = scaled_patch[watermarking_mask]
    elif args.w_injection == "seed":
        init_latents_w[watermarking_mask] = scaled_patch[watermarking_mask]
        return init_latents_w.to(orig_dtype)
    init_latents_w = torch.fft.ifft2(torch.fft.ifftshift(init_latents_w_fft, dim=(-1, -2))).real
    return init_latents_w.to(orig_dtype)

# Optimization #5: Ensemble detection
def eval_watermark_ensemble(rev_no_w, rev_w, mask, gt_patch, args):
    if args.w_channel != -1:
        return _eval_base(rev_no_w, rev_w, mask, gt_patch, args)
    rev_no_fft = torch.fft.fftshift(torch.fft.fft2(rev_no_w.float()), dim=(-1, -2))
    rev_w_fft = torch.fft.fftshift(torch.fft.fft2(rev_w.float()), dim=(-1, -2))
    no_w_ch, w_ch = [], []
    for ch in range(rev_no_w.shape[1]):
        cm = mask[:, ch:ch+1]
        if not cm.any(): continue
        no_w_ch.append(torch.abs(rev_no_fft[:, ch:ch+1][cm] - gt_patch[:, ch:ch+1][cm]).mean().item())
        w_ch.append(torch.abs(rev_w_fft[:, ch:ch+1][cm] - gt_patch[:, ch:ch+1][cm]).mean().item())
    if not w_ch: return _eval_base(rev_no_w, rev_w, mask, gt_patch, args)
    return float(np.mean(no_w_ch)), float(np.mean(w_ch))

def build_args(config, attack):
    a = argparse.Namespace()
    a.model_id=config.model_id; a.image_length=config.image_length; a.num_images=config.num_images
    a.guidance_scale=config.guidance_scale; a.num_inference_steps=config.num_inference_steps
    a.test_num_inference_steps=config.test_num_inference_steps; a.gen_seed=config.gen_seed
    a.start=config.start; a.end=config.end; a.run_name=f"{config.name}_{attack.name}"
    a.with_tracking=False; a.max_num_log_image=100
    wm=config.watermark
    a.w_seed=wm.w_seed; a.w_channel=wm.w_channel; a.w_pattern=wm.w_pattern
    a.w_mask_shape=wm.w_mask_shape; a.w_radius=wm.w_radius; a.w_measurement=wm.w_measurement
    a.w_injection=wm.w_injection; a.w_pattern_const=wm.w_pattern_const
    a.r_degree=attack.r_degree; a.jpeg_ratio=attack.jpeg_ratio; a.crop_scale=attack.crop_scale
    a.crop_ratio=attack.crop_ratio; a.gaussian_blur_r=attack.gaussian_blur_r
    a.gaussian_std=attack.gaussian_std; a.brightness_factor=attack.brightness_factor; a.rand_aug=attack.rand_aug
    a.dataset="Gustavosta/Stable-Diffusion-Prompts"
    a.reference_model=config.reference_model; a.reference_model_pretrain=config.reference_model_pretrain
    return a

def load_ckpt(p):
    if p.exists():
        with open(p) as f: return json.load(f)
    return None

def save_ckpt(p, data):
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p,"w") as f: json.dump(data, f, indent=2)

def run_optimized(config, skip_clip=False, scale_factor=1.1):
    device = get_device()
    dtype = torch.float16

    print(f"\\n{\'=\'*60}\\nOPTIMIZED: {config.name} | Device: {device} | Images: {config.start}-{config.end}")
    print(f"Watermark: ch={config.watermark.w_channel}, pattern={config.watermark.w_pattern}, r={config.watermark.w_radius}")
    print(f"Opts: scale={scale_factor}, inv_steps={config.test_num_inference_steps}\\n{\'=\'*60}\\n")

    base_dir = Path(__file__).resolve().parent
    ckpt_dir = base_dir/config.checkpoint_dir/config.name
    res_dir = base_dir/config.results_dir/config.name
    out_dir = base_dir/config.outputs_dir/config.name
    for d in [ckpt_dir, res_dir, out_dir]: d.mkdir(parents=True, exist_ok=True)

    args = build_args(config, config.attacks[0] if config.attacks else AttackConfig())

    print("Loading Stable Diffusion pipeline...")
    scheduler = DPMSolverMultistepScheduler.from_pretrained(args.model_id, subfolder="scheduler")
    pipe = InversableStableDiffusionPipeline.from_pretrained(args.model_id, scheduler=scheduler, torch_dtype=dtype)
    pipe = pipe.to(device)
    try: pipe.enable_xformers_memory_efficient_attention(); print("  xformers enabled")
    except: pass

    ref_model=ref_clip_preprocess=ref_tokenizer=None
    if not skip_clip and config.reference_model:
        print(f"Loading CLIP: {config.reference_model}...")
        import open_clip
        ref_model, _, ref_clip_preprocess = open_clip.create_model_and_transforms(config.reference_model, pretrained=config.reference_model_pretrain, device=device)
        ref_tokenizer = open_clip.get_tokenizer(config.reference_model)

    text_embeddings = pipe.get_text_embedding("")
    gt_patch = get_watermarking_pattern(pipe, args, device)

    all_results = {}
    for attack in config.attacks:
        args_a = build_args(config, attack)
        ckpt_path = ckpt_dir/f"{attack.name}_checkpoint.json"
        ckpt = load_ckpt(ckpt_path)

        if ckpt and ckpt.get("completed"):
            print(f"  {attack.name}: already done, loading.")
            all_results[attack.name] = ckpt["final_results"]; continue

        start_idx=config.start; results=[]; no_w_metrics=[]; w_metrics=[]; clips=[]; clips_w=[]; saved=[]
        if ckpt and not ckpt.get("completed"):
            start_idx=ckpt["last_idx"]+1; results=ckpt.get("results",[]); no_w_metrics=ckpt.get("no_w_metrics",[])
            w_metrics=ckpt.get("w_metrics",[]); clips=ckpt.get("clips",[]); clips_w=ckpt.get("clips_w",[]); saved=ckpt.get("saved",[])
            print(f"  {attack.name}: resuming from {start_idx}")

        t0=time.time()
        for i in tqdm(range(start_idx, config.end), desc=f"  {attack.name}"):
            seed=i+config.gen_seed; prompt=get_prompt(i)
            set_random_seed(seed)
            lat_no_w=pipe.get_random_latents()
            out_no_w=pipe(prompt,num_images_per_prompt=1,guidance_scale=args_a.guidance_scale,num_inference_steps=args_a.num_inference_steps,height=args_a.image_length,width=args_a.image_length,latents=lat_no_w)
            img_no_w=out_no_w.images[0]

            lat_w=copy.deepcopy(lat_no_w)
            mask=get_watermarking_mask(lat_w, args_a, device)
            lat_w=inject_watermark_scaled(lat_w, mask, gt_patch, args_a, scale_factor=scale_factor)
            out_w=pipe(prompt,num_images_per_prompt=1,guidance_scale=args_a.guidance_scale,num_inference_steps=args_a.num_inference_steps,height=args_a.image_length,width=args_a.image_length,latents=lat_w)
            img_w=out_w.images[0]

            if len(saved)<5:
                d=out_dir/attack.name; d.mkdir(parents=True,exist_ok=True)
                img_no_w.save(d/f"img_{i:04d}_no_wm.png"); img_w.save(d/f"img_{i:04d}_wm.png"); saved.append(i)

            img_no_w_a, img_w_a = image_distortion(img_no_w, img_w, seed, args_a)

            t_no=transform_img(img_no_w_a).unsqueeze(0).to(text_embeddings.dtype).to(device)
            rev_no=pipe.forward_diffusion(latents=pipe.get_image_latents(t_no,sample=False),text_embeddings=text_embeddings,guidance_scale=1,num_inference_steps=args_a.test_num_inference_steps)
            t_w=transform_img(img_w_a).unsqueeze(0).to(text_embeddings.dtype).to(device)
            rev_w=pipe.forward_diffusion(latents=pipe.get_image_latents(t_w,sample=False),text_embeddings=text_embeddings,guidance_scale=1,num_inference_steps=args_a.test_num_inference_steps)

            nm,wm_=eval_watermark_ensemble(rev_no,rev_w,mask,gt_patch,args_a)
            cs_no=cs_w=0.0
            if ref_model:
                sims=measure_similarity([img_no_w,img_w],prompt,ref_model,ref_clip_preprocess,ref_tokenizer,device)
                cs_no=sims[0].item(); cs_w=sims[1].item()

            results.append({"i":i,"prompt":prompt,"no_w":nm,"w":wm_,"clip_no":cs_no,"clip_w":cs_w})
            no_w_metrics.append(-nm); w_metrics.append(-wm_); clips.append(cs_no); clips_w.append(cs_w)

            if (i-config.start+1)%config.checkpoint_every==0 or i==config.end-1:
                save_ckpt(ckpt_path,{"attack":attack.name,"last_idx":i,"completed":False,"results":results,
                    "no_w_metrics":no_w_metrics,"w_metrics":w_metrics,"clips":clips,"clips_w":clips_w,"saved":saved})

        elapsed=time.time()-t0
        preds=no_w_metrics+w_metrics; labels=[0]*len(no_w_metrics)+[1]*len(w_metrics)
        fpr,tpr,_=metrics.roc_curve(labels,preds,pos_label=1)
        auc=metrics.auc(fpr,tpr); acc=np.max(1-(fpr+(1-tpr))/2)
        low=tpr[np.where(fpr<.01)[0][-1]] if len(np.where(fpr<.01)[0])>0 else 0.0

        final={"attack":attack.name,"approach":"optimized","num_images":config.end-config.start,
            "auc":float(auc),"acc":float(acc),"tpr_at_1fpr":float(low),
            "mean_no_w_metric":float(mean([-m for m in no_w_metrics])),"mean_w_metric":float(mean([-m for m in w_metrics])),
            "clip_score_mean":float(mean(clips)) if clips and any(c!=0 for c in clips) else None,
            "clip_score_w_mean":float(mean(clips_w)) if clips_w and any(c!=0 for c in clips_w) else None,
            "elapsed_seconds":elapsed,
            "watermark_params":{"w_channel":config.watermark.w_channel,"w_pattern":config.watermark.w_pattern,"w_radius":config.watermark.w_radius},
            "optimizations":{"scale_factor":scale_factor,"test_num_inference_steps":config.test_num_inference_steps,
                "multi_channel":config.watermark.w_channel==-1,"ensemble_detection":config.watermark.w_channel==-1}}
        all_results[attack.name]=final
        save_ckpt(ckpt_path,{"attack":attack.name,"last_idx":config.end-1,"completed":True,"results":results,
            "no_w_metrics":no_w_metrics,"w_metrics":w_metrics,"clips":clips,"clips_w":clips_w,"saved":saved,"final_results":final})
        print(f"  AUC:{auc:.4f} Acc:{acc:.4f} TPR@1%FPR:{low:.4f} Time:{elapsed:.1f}s")

    with open(res_dir/"all_attacks_results.json","w") as f: json.dump(all_results,f,indent=2)
    print(f"\\nOptimized results: {res_dir/\'all_attacks_results.json\'}")
    return all_results

if __name__=="__main__":
    import argparse as ap; p=ap.ArgumentParser(); p.add_argument("--scale",default="small"); p.add_argument("--skip_clip",action="store_true"); p.add_argument("--scale_factor",type=float,default=1.1); a=p.parse_args()
    from configs import get_small_scale_optimized, get_large_scale_optimized
    run_optimized(get_small_scale_optimized() if a.scale=="small" else get_large_scale_optimized(), skip_clip=a.skip_clip, scale_factor=a.scale_factor)
''')

print("  run_optimized.py written")

In [ ]:
# ===== compare_results.py =====
with open("experiment/compare_results.py", "w") as f:
    f.write('''"""Compare baseline vs optimized results. Prints table, saves CSV + plots."""
import json, csv
from pathlib import Path

def load_results(d):
    p = d/"all_attacks_results.json"
    if not p.exists(): return {}
    with open(p) as f: return json.load(f)

def fmt(v, f=".4f"):
    if v is None: return "N/A"
    if isinstance(v, float): return f"{v:{f}}"
    return str(v)

def compare(scale="small"):
    base = Path(__file__).resolve().parent/"results"
    bl = load_results(base/f"{scale}_baseline")
    op = load_results(base/f"{scale}_optimized")
    if not bl and not op: print(f"No results for {scale}. Run experiments first."); return

    print(f"\\n{\'=\'*100}\\n  COMPARISON: BASELINE vs OPTIMIZED  ({scale.upper()} SCALE)\\n{\'=\'*100}")
    print(f"\\n{\'Attack\':<22} | {\'Metric\':<16} | {\'Baseline\':>10} | {\'Optimized\':>10} | {\'Delta\':>10} | {\'Winner\':>8}")
    print("-"*100)

    attacks = sorted(set(list(bl.keys())+list(op.keys())))
    ms = [("auc","AUC",True),("acc","Accuracy",True),("tpr_at_1fpr","TPR@1%FPR",True),
          ("mean_w_metric","W Metric",False),("clip_score_w_mean","CLIP (W)",True)]
    summary = {"opt":0,"base":0,"tie":0}

    for atk in attacks:
        b,o = bl.get(atk,{}), op.get(atk,{})
        for mk,mn,hb in ms:
            bv,ov = b.get(mk), o.get(mk)
            if bv is None and ov is None: continue
            bs,os_ = fmt(bv), fmt(ov)
            if bv is not None and ov is not None:
                d=ov-bv; ds=f"{d:+.4f}"
                w="OPT" if (d>0.001 if hb else d<-0.001) else ("BASE" if (d<-0.001 if hb else d>0.001) else "TIE")
                summary["opt" if w=="OPT" else "base" if w=="BASE" else "tie"]+=1
            else: ds="N/A"; w="N/A"
            print(f"{atk:<22} | {mn:<16} | {bs:>10} | {os_:>10} | {ds:>10} | {w:>8}")
        bt,ot = b.get("elapsed_seconds"), o.get("elapsed_seconds")
        if bt and ot: print(f"{atk:<22} | {\'Time (s)\':<16} | {bt:>10.1f} | {ot:>10.1f} | {bt/ot if ot else 0:>9.2f}x | {\'---\':>8}")
        print("-"*100)

    print(f"\\n{\'=\'*60}\\n  SUMMARY\\n{\'=\'*60}")
    print(f"  Optimized wins: {summary[\'opt\']}\\n  Baseline wins:  {summary[\'base\']}\\n  Ties:           {summary[\'tie\']}")
    if bl:
        p=next(iter(bl.values())).get("watermark_params",{})
        print(f"\\n  Baseline:  ch={p.get(\'w_channel\')}, pattern={p.get(\'w_pattern\')}, radius={p.get(\'w_radius\')}")
    if op:
        p=next(iter(op.values())).get("watermark_params",{}); o_=next(iter(op.values())).get("optimizations",{})
        print(f"  Optimized: ch={p.get(\'w_channel\')}, pattern={p.get(\'w_pattern\')}, radius={p.get(\'w_radius\')}")
        print(f"  Opts:      scale={o_.get(\'scale_factor\')}, inv_steps={o_.get(\'test_num_inference_steps\')}, multi_ch={o_.get(\'multi_channel\')}")

    csv_path = base/f"{scale}_comparison.csv"
    with open(csv_path,"w",newline="") as f:
        w=csv.writer(f)
        w.writerow(["Attack","Approach","AUC","Accuracy","TPR@1%FPR","Mean_NoW","Mean_W","CLIP_NoW","CLIP_W","Time_s"])
        for atk in attacks:
            for ap_,res in [("baseline",bl),("optimized",op)]:
                r=res.get(atk,{})
                if not r: continue
                w.writerow([atk,ap_,fmt(r.get("auc")),fmt(r.get("acc")),fmt(r.get("tpr_at_1fpr")),
                    fmt(r.get("mean_no_w_metric")),fmt(r.get("mean_w_metric")),
                    fmt(r.get("clip_score_mean")),fmt(r.get("clip_score_w_mean")),fmt(r.get("elapsed_seconds"),".1f")])
    print(f"\\n  CSV: {csv_path}")

    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        pd = base/f"{scale}_plots"; pd.mkdir(parents=True, exist_ok=True)
        for mk,mn in [("auc","AUC"),("acc","Accuracy"),("tpr_at_1fpr","TPR@1%FPR")]:
            bv=[bl.get(a,{}).get(mk,0) or 0 for a in attacks]
            ov=[op.get(a,{}).get(mk,0) or 0 for a in attacks]
            x=range(len(attacks)); w_=0.35
            fig,ax=plt.subplots(figsize=(max(10,len(attacks)*1.5),6))
            ax.bar([i-w_/2 for i in x],bv,w_,label="Baseline",color="#4C72B0")
            ax.bar([i+w_/2 for i in x],ov,w_,label="Optimized",color="#DD8452")
            ax.set_ylabel(mn); ax.set_title(f"{mn}: Baseline vs Optimized")
            ax.set_xticks(list(x)); ax.set_xticklabels(attacks,rotation=45,ha="right")
            ax.legend(); ax.set_ylim(0,1.05); plt.tight_layout()
            plt.savefig(pd/f"comparison_{mk}.png",dpi=150); plt.close()
        print(f"  Plots: {pd}")
    except ImportError: print("  matplotlib not available, skipping plots.")
    print()

if __name__=="__main__":
    import argparse; p=argparse.ArgumentParser(); p.add_argument("--scale",default="small"); a=p.parse_args()
    if a.scale=="both": compare("small"); compare("large")
    else: compare(a.scale)
''')

print("  compare_results.py written")

In [ ]:
# Create __init__.py so experiment/ is a package
with open("experiment/__init__.py", "w") as f:
    f.write("")

print("All experiment files written!")
print("Files:")
import os
for f in sorted(os.listdir("experiment")):
    print(f"  experiment/{f}")

---
## Cell 5: Pre-download Model

Downloads Stable Diffusion v1.5 (~2GB). This only runs once — subsequent runs use the cache.

In [ ]:
import sys
sys.path.insert(0, "tree-ring-watermark")
from diffusers import DPMSolverMultistepScheduler
from inverse_stable_diffusion import InversableStableDiffusionPipeline
import torch

MODEL_ID = "runwayml/stable-diffusion-v1-5"
print(f"Downloading {MODEL_ID}...")
scheduler = DPMSolverMultistepScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")
pipe = InversableStableDiffusionPipeline.from_pretrained(
    MODEL_ID, scheduler=scheduler, torch_dtype=torch.float16
)
print("Model downloaded and cached.")
del pipe, scheduler  # free memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## Cell 6: Run BASELINE Experiment

In [ ]:
import sys, importlib
sys.path.insert(0, "experiment")
sys.path.insert(0, "tree-ring-watermark")

import experiment.configs as configs
import experiment.run_baseline as run_baseline_mod
importlib.reload(configs)
importlib.reload(run_baseline_mod)

if SCALE == "small":
    config = configs.get_small_scale_baseline()
else:
    config = configs.get_large_scale_baseline()

print(f"Running BASELINE ({SCALE} scale)...")
baseline_results = run_baseline_mod.run_baseline(config, skip_clip=SKIP_CLIP)

---
## Cell 7: Run OPTIMIZED Experiment

In [ ]:
import experiment.run_optimized as run_optimized_mod
importlib.reload(run_optimized_mod)

if SCALE == "small":
    config_opt = configs.get_small_scale_optimized()
else:
    config_opt = configs.get_large_scale_optimized()

print(f"Running OPTIMIZED ({SCALE} scale)...")
optimized_results = run_optimized_mod.run_optimized(config_opt, skip_clip=SKIP_CLIP, scale_factor=SCALE_FACTOR)

---
## Cell 8: Compare Results

In [ ]:
import experiment.compare_results as compare_mod
importlib.reload(compare_mod)

compare_mod.compare(SCALE)

---
## Cell 9: Display Comparison Plots

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plots_dir = Path("experiment/results") / f"{SCALE}_plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"\n{p.name}")
        display(Image(filename=str(p), width=800))
else:
    print("No plots found. Check if matplotlib is installed.")

---
## Cell 10: Display Sample Images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from pathlib import Path

for approach in ["baseline", "optimized"]:
    out_dir = Path("experiment/outputs") / f"{SCALE}_{approach}" / "no_attack"
    if not out_dir.exists():
        print(f"No output images for {approach}")
        continue

    wm_imgs = sorted(out_dir.glob("*_wm.png"))[:3]
    no_wm_imgs = sorted(out_dir.glob("*_no_wm.png"))[:3]

    if not wm_imgs:
        continue

    n = min(len(wm_imgs), 3)
    fig, axes = plt.subplots(2, n, figsize=(5*n, 10))
    if n == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle(f"{approach.upper()} - No Attack", fontsize=16)

    for j in range(n):
        axes[0][j].imshow(PILImage.open(no_wm_imgs[j]))
        axes[0][j].set_title("No Watermark")
        axes[0][j].axis("off")
        axes[1][j].imshow(PILImage.open(wm_imgs[j]))
        axes[1][j].set_title("Watermarked")
        axes[1][j].axis("off")

    plt.tight_layout()
    plt.show()

---
## Cell 11: Save Results to Google Drive (Optional)

Run this cell to persist results across Colab sessions.

In [ ]:
#@title Save to Google Drive {display-mode: "form"}
SAVE_TO_DRIVE = False  #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    drive_dir = Path("/content/drive/MyDrive/tree_ring_results")
    drive_dir.mkdir(parents=True, exist_ok=True)

    # Copy results, checkpoints, outputs
    for folder in ["results", "checkpoints", "outputs"]:
        src = Path("experiment") / folder
        dst = drive_dir / folder
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"  Saved {folder} -> {dst}")

    print(f"\nAll results saved to: {drive_dir}")
else:
    print("Skipping Google Drive save. Set SAVE_TO_DRIVE = True to enable.")